# Wi-Fi Fingerprint Indoor Localization — Regression Track

**Objective:** Predict the indoor (X, Y) coordinates of a device using Wi-Fi RSSI fingerprints.

**Dataset:** UJIndoorLoc (UCI ML Repository) — 520 WAP signal readings from 3 buildings across multiple floors.

**Team:** K Ganesh Giridhar (519) · G R Balaji (510) · A Suhas Reddy (503)

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully")

## 2. Dataset Loading & Audit

Load the raw UJIndoorLoc training and validation sets. We keep the originals untouched in `data/raw/`.

In [ ]:
train_df = pd.read_csv('../data/raw/trainingData.csv')
val_df   = pd.read_csv('../data/raw/validationData.csv')

print(f"Training set  : {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Validation set: {val_df.shape[0]} rows, {val_df.shape[1]} columns")

**Observation:** Training set has ~19,937 samples and 529 columns (520 WAPs + 9 metadata). Validation has 1,111 samples.

In [ ]:
# Column types overview
train_df.info(verbose=False)

In [ ]:
# First few rows — WAP columns and target columns
print("WAP columns sample:")
print(train_df.iloc[:3, :5])
print()
print("Target / metadata columns:")
print(train_df[['LONGITUDE','LATITUDE','FLOOR','BUILDINGID','SPACEID']].head())

In [ ]:
# Check for missing values
missing = train_df.isnull().sum().sum()
print(f"Total missing values in training set: {missing}")

**Inference:** No traditional missing values (NaN) exist. However, the RSSI value `+100` is a **sentinel** meaning 'AP not detected' — this needs special treatment later.

In [ ]:
# WAP columns
wap_cols = [c for c in train_df.columns if c.startswith('WAP')]
print(f"Number of WAP features: {len(wap_cols)}")

# Check how many +100 values exist
sentinel_count = (train_df[wap_cols] == 100).sum().sum()
total_cells = train_df[wap_cols].shape[0] * train_df[wap_cols].shape[1]
pct = (sentinel_count / total_cells) * 100
print(f"Sentinel (+100) values: {sentinel_count:,} out of {total_cells:,} ({pct:.1f}%)")

**Key Finding:** A very large proportion of WAP readings are +100 (not detected). This is expected — a device can only see a small subset of all 520 access points from any location.

In [ ]:
# Target variable distributions
print("=== LONGITUDE ===")
print(train_df['LONGITUDE'].describe())
print()
print("=== LATITUDE ===")
print(train_df['LATITUDE'].describe())
print()
print("=== FLOOR distribution ===")
print(train_df['FLOOR'].value_counts().sort_index())
print()
print("=== BUILDING distribution ===")
print(train_df['BUILDINGID'].value_counts().sort_index())

**Observation:**
- Coordinates span a wide range (campus-level) across 3 buildings
- Floor values range from 0 to 4 (5 floors total)
- Building 0, 1, 2 have varying sample counts — slightly imbalanced but acceptable

---
## 3. Exploratory Data Analysis

### 3.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Longitude
axes[0].hist(train_df['LONGITUDE'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Longitude Distribution')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Count')

# Latitude
axes[1].hist(train_df['LATITUDE'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Latitude Distribution')
axes[1].set_xlabel('Latitude')
axes[1].set_ylabel('Count')

# Floor
train_df['FLOOR'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='seagreen', edgecolor='black')
axes[2].set_title('Floor Distribution')
axes[2].set_xlabel('Floor')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

**Inference:** Longitude and Latitude show multi-modal distributions — samples are clustered around specific building regions. Floor 0, 1, 2 have more samples while floors 3 and 4 are less represented.

### 3.2 Spatial Distribution (Scatter Plot)

In [ ]:
plt.figure(figsize=(10, 8))
scatter = plt.scatter(train_df['LONGITUDE'], train_df['LATITUDE'],
                      c=train_df['BUILDINGID'], cmap='Set1', alpha=0.4, s=10)
plt.colorbar(scatter, label='Building ID')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Spatial Distribution of Samples by Building')
plt.tight_layout()
plt.show()

**Inference:** The three buildings are clearly separated in coordinate space. This is good — the model can learn distinct spatial patterns for each building.

### 3.3 Feature-Target Scatter Plots

In [ ]:
# Find the top 5 most frequently detected WAPs (least +100 values)
detected_counts = (train_df[wap_cols] != 100).sum().sort_values(ascending=False)
top_waps = detected_counts.head(5).index.tolist()
print("Top 5 most commonly detected WAPs:", top_waps)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# WAP vs Longitude
wap_example = top_waps[0]
mask = train_df[wap_example] != 100
axes[0].scatter(train_df.loc[mask, wap_example], train_df.loc[mask, 'LONGITUDE'],
                alpha=0.3, s=8, color='steelblue')
axes[0].set_xlabel(f'{wap_example} RSSI')
axes[0].set_ylabel('Longitude')
axes[0].set_title(f'{wap_example} vs Longitude')

# WAP vs Latitude
axes[1].scatter(train_df.loc[mask, wap_example], train_df.loc[mask, 'LATITUDE'],
                alpha=0.3, s=8, color='coral')
axes[1].set_xlabel(f'{wap_example} RSSI')
axes[1].set_ylabel('Latitude')
axes[1].set_title(f'{wap_example} vs Latitude')

plt.tight_layout()
plt.show()

**Inference:** Individual WAPs show localized detection patterns — strong signal from a particular AP correlates with proximity to it. This confirms RSSI values carry useful spatial information.

### 3.4 Correlation Heatmap (Top Detected WAPs)

In [ ]:
# Correlation among top 15 WAPs and targets
top15 = detected_counts.head(15).index.tolist()
corr_cols = top15 + ['LONGITUDE', 'LATITUDE']

# Replace 100 with NaN for correlation calculation
corr_df = train_df[corr_cols].replace(100, np.nan)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Top 15 WAPs + Targets')
plt.tight_layout()
plt.show()

**Inference:**
- Some WAPs show moderate correlation with LONGITUDE or LATITUDE — these will be important features
- Many WAPs are weakly correlated with each other — low multicollinearity is good
- The heatmap confirms that WAP signals encode spatial information differently per AP

### 3.5 Floor-wise Coordinate Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, bid in enumerate(sorted(train_df['BUILDINGID'].unique())):
    bdata = train_df[train_df['BUILDINGID'] == bid]
    scatter = axes[i].scatter(bdata['LONGITUDE'], bdata['LATITUDE'],
                              c=bdata['FLOOR'], cmap='viridis', alpha=0.5, s=8)
    axes[i].set_title(f'Building {bid}')
    axes[i].set_xlabel('Longitude')
    axes[i].set_ylabel('Latitude')
    plt.colorbar(scatter, ax=axes[i], label='Floor')

plt.suptitle('Coordinate Distribution by Building and Floor', y=1.02)
plt.tight_layout()
plt.show()

**Inference:** Within each building, different floors overlap in X-Y space but are vertically separated. This means coordinate regression alone won't identify the floor — we need a separate classifier for that (Classification track).

---
## 4. Data Cleaning

### 4.1 Sentinel Value Treatment

RSSI = +100 means the AP was not detected. We replace it with **-105 dBm** (below the typical detection threshold of -104 dBm). This preserves the ordinal relationship: weaker signals have more negative values.

In [ ]:
# Replace sentinel +100 with -105
train_clean = train_df.copy()
val_clean = val_df.copy()

train_clean[wap_cols] = train_clean[wap_cols].replace(100, -105)
val_clean[wap_cols] = val_clean[wap_cols].replace(100, -105)

print("Sentinel replacement done")
print(f"RSSI range now: [{train_clean[wap_cols].min().min()}, {train_clean[wap_cols].max().max()}]")

**Why -105?** The weakest detectable signal is around -104 dBm. Setting undetected APs to -105 keeps them below the detection floor without introducing a massive gap (like -999 would).

### 4.2 Remove Zero-Variance WAPs

Some WAPs may never be detected in the training set — they have constant values. These add no information.

In [ ]:
# Find WAPs with zero variance (all same value after cleaning)
from sklearn.feature_selection import VarianceThreshold

variances = train_clean[wap_cols].var()
zero_var = variances[variances == 0].index.tolist()
print(f"WAPs with zero variance: {len(zero_var)}")

# Remove them
wap_cols_clean = [c for c in wap_cols if c not in zero_var]
print(f"WAPs remaining after removal: {len(wap_cols_clean)}")

**Decision:** Removing zero-variance WAPs reduces dimensionality without losing any information. These APs were never detected by any sample.

### 4.3 Duplicate Check

In [ ]:
dupes = train_clean.duplicated().sum()
print(f"Duplicate rows: {dupes}")

if dupes > 0:
    train_clean = train_clean.drop_duplicates()
    print(f"After removal: {train_clean.shape[0]} rows")
else:
    print("No duplicates found — good")

### 4.4 Outlier Check

In [ ]:
# Check target variable outliers using IQR
for col in ['LONGITUDE', 'LATITUDE']:
    Q1 = train_clean[col].quantile(0.25)
    Q3 = train_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((train_clean[col] < lower) | (train_clean[col] > upper)).sum()
    print(f"{col}: {outliers} outliers (IQR method)")

**Decision:** We keep the coordinate outliers since they represent real physical locations at the edges of the buildings. Removing them would lose valid spatial data points.

---
## 5. Feature Engineering

We create summary features from the raw RSSI fingerprint. These capture the overall signal environment at each location.

In [ ]:
# Engineered features based on RSSI values
# Using -105 as 'not detected' threshold

def engineer_features(df, wap_columns):
    """Create RSSI summary features from WAP readings."""
    wap_data = df[wap_columns]

    # Count of detected APs (RSSI > -105)
    df['n_visible_aps'] = (wap_data > -105).sum(axis=1)

    # Mean RSSI of detected APs only
    detected = wap_data.replace(-105, np.nan)
    df['mean_rssi'] = detected.mean(axis=1)

    # Max (strongest) signal
    df['max_rssi'] = detected.max(axis=1)

    # Std of detected signals
    df['std_rssi'] = detected.std(axis=1)

    # Range of detected signals
    df['rssi_range'] = detected.max(axis=1) - detected.min(axis=1)

    return df

train_clean = engineer_features(train_clean, wap_cols_clean)
val_clean = engineer_features(val_clean, wap_cols_clean)

print("Engineered features created:")
print(train_clean[['n_visible_aps','mean_rssi','max_rssi','std_rssi','rssi_range']].describe().round(2))

**Why these features?**
- `n_visible_aps`: How many APs a device can see varies by location — open areas see more APs
- `mean_rssi`: Overall signal strength indicates proximity to AP clusters
- `max_rssi`: The strongest signal usually comes from the nearest AP
- `std_rssi` and `rssi_range`: Signal variability captures whether the device is near many APs or just a few

These give the model a high-level summary alongside the detailed per-AP readings.

In [ ]:
# Check correlation of new features with targets
eng_features = ['n_visible_aps', 'mean_rssi', 'max_rssi', 'std_rssi', 'rssi_range']

print("Correlation with LONGITUDE:")
for f in eng_features:
    corr = train_clean[f].corr(train_clean['LONGITUDE'])
    print(f"  {f}: {corr:.3f}")
print()
print("Correlation with LATITUDE:")
for f in eng_features:
    corr = train_clean[f].corr(train_clean['LATITUDE'])
    print(f"  {f}: {corr:.3f}")

**Inference:** The engineered features show some correlation with coordinates. Even modest correlations can help — they provide the model with aggregate spatial cues.

---
## 6. Train-Test Split & Scaling

**Important:** We fit the scaler on the training set only. Fitting on the full dataset would be data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Feature columns = WAP columns + engineered features
feature_cols = wap_cols_clean + eng_features

X = train_clean[feature_cols]
y_lon = train_clean['LONGITUDE']
y_lat = train_clean['LATITUDE']

# 80:20 split
X_train, X_test, y_lon_train, y_lon_test = train_test_split(
    X, y_lon, test_size=0.2, random_state=42
)

# Same split indices for latitude
_, _, y_lat_train, y_lat_test = train_test_split(
    X, y_lat, test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Testing:  {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")

In [ ]:
# Scale features — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done (fitted on training set only)")
print(f"Train mean ≈ {X_train_scaled.mean():.6f} (should be ~0)")
print(f"Train std  ≈ {X_train_scaled.std():.4f}  (should be ~1)")

**Why StandardScaler?** Algorithms like SVR, KNN, and regularized linear models (Ridge, Lasso) are sensitive to feature scale. StandardScaler centers features at mean=0 and std=1 without distorting the RSSI value distribution.

---
## 7. Regression Models

We train all 10 required regression algorithms on the same train/test split. For simplicity we predict **LONGITUDE** as the primary target and report all metrics. The same pipeline applies to LATITUDE.

> **Evaluation metrics:** R², RMSE, MAE

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    """Train, predict, and return metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    r2   = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)

    print(f"{model_name:35s} | R²: {r2:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f}")
    return {'Model': model_name, 'R2': round(r2, 4), 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'predictions': y_pred}

results = []

### 7.1 Linear Regression

Baseline model — fits a straight hyperplane through the feature space.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
res = evaluate_model(lr, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Linear Regression')
results.append(res)

**Inference:** Linear Regression gives us a baseline R². If it's already high, the relationship between WAP signals and coordinates is fairly linear. If low, we need non-linear models.

### 7.2 Ridge Regression

L2 regularisation — penalises large coefficients to reduce overfitting, especially useful with 500+ features.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0, random_state=42)
res = evaluate_model(ridge, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Ridge Regression')
results.append(res)

**Inference:** Ridge typically performs similar to or slightly better than plain Linear Regression when features are many and possibly correlated. The L2 penalty keeps coefficients small.

### 7.3 Lasso Regression

L1 regularisation — can drive some coefficients to exactly zero, effectively doing feature selection.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1, random_state=42, max_iter=5000)
res = evaluate_model(lasso, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Lasso Regression')
results.append(res)

# How many features got zeroed out?
n_zero = (lasso.coef_ == 0).sum()
print(f"Features with zero coefficient: {n_zero} out of {len(lasso.coef_)}")

**Inference:** Lasso's feature sparsity tells us how many WAPs are truly relevant for predicting longitude. If many coefficients are zero, most APs don't contribute to positioning from this direction.

### 7.4 ElasticNet Regression

Combines L1 and L2 penalties — a middle ground between Ridge and Lasso.

In [ ]:
from sklearn.linear_model import ElasticNet

enet = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=5000)
res = evaluate_model(enet, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'ElasticNet Regression')
results.append(res)

**Inference:** ElasticNet balances feature selection (L1) with coefficient shrinkage (L2). Useful when features are grouped — correlated WAPs from the same area get shared weight instead of one being zeroed.

### 7.5 Polynomial Regression

Apply polynomial feature transformation then fit linear regression. We use degree=2 as degree=3 with 500+ features would be computationally expensive.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Use only top 20 features to keep polynomial expansion manageable
from sklearn.feature_selection import SelectKBest, f_regression

poly_pipe = Pipeline([
    ('select', SelectKBest(f_regression, k=20)),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('reg', LinearRegression())
])

res = evaluate_model(poly_pipe, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Polynomial Regression (deg=2)')
results.append(res)

**Inference:** Polynomial features capture non-linear relationships. We select top-20 features first to avoid the combinatorial explosion of polynomial terms with 500+ features. This is a practical trade-off between expressiveness and tractability.